# 02 · Ecuaciones transcritas y dato faltante

## Objetivo
Mostrar la expresión que el código usa para la gravedad específica y para el PIP en pies, y el estado que devuelve si falta BS&W.

## Alcance
Las fórmulas viven en `src/modelo_deterministico/`. Este cuaderno imprime su documentación y un caso construido. No resuelve el caudal desde la potencia y no ejecuta el XLSM.


In [1]:
from pathlib import Path
import os, sys
aqui = Path.cwd().resolve()
expediente = next(c for c in [aqui, *aqui.parents] if (c / "dataset.md").is_file() and (c / "entrada").is_dir())
os.chdir(expediente)
if str(expediente / "src") not in sys.path:
    sys.path.insert(0, str(expediente / "src"))
libro = expediente / "entrada" / "BES_jun.xlsx"

from ingesta.leer_bes_jun import leer_libro
import modelo_deterministico.fluidos_bes_jun as fluidos
import modelo_deterministico.balance_bes_jun as balance
print(fluidos.gravedad_especifica_api.__doc__)
print(balance.pip_en_ft.__doc__)
hoja = leer_libro(libro)[0]
faltante = fluidos.evaluar_fluido(hoja["api_deg"], hoja["temperatura_c"], None, hoja["densidad_agua_lb_gal"])
print("caso_construido", hoja["hoja"], "bsw_forzado", None, faltante.estado, faltante.motivo)
assert faltante.estado == "no_calculable"
assert faltante.resultados == {}


Gravedad específica del crudo a 60 °F.

Fórmula F-01: gamma = 141.5 / (131.5 + API).
Fuente: «Copia de FINAL BES-jun.xlsx», celda de «Gravedad Oil @ 60ºF»
(p. ej. QF-554H!B96) y XLSM J4.
Entrada: api_deg en °API. Salida: adimensional.

PIP en pies de columna, con el signo de la hoja.

(P_sensor - ((TVD_intake - TVD_sensor) * gradiente)) / gradiente
Fuente: QF-554H!G96 y la celda «PIP en (ft)» de cada hoja.



caso_construido QF-554H bsw_forzado None no_calculable Faltan entradas esenciales del fluido: bsw_numero


## Decisión de uso
F-01, `141.5 / (131.5 + °API)`, está implementada. La entrada es °API y la salida es adimensional. En las cinco hojas el API cacheado es 12.

`pip_en_ft` usa la presión de sensor en psi, las TVD de intake y de sensor en ft, y el gradiente en psi/ft. El signo es el de la celda de la hoja. No es la estimación sin sensor del documento de ecuaciones.

## Caso que no se calcula
El ejemplo toma API, temperatura y densidad de agua de la primera hoja, QF-554H, y pone BS&W en blanco. Es un caso construido, no una sexta hoja. El estado es `no_calculable` y el motivo nombra `bsw_numero`. No se rellena ese hueco.

La puerta de fórmulas del plan sigue en borrador. Transcribir estas celdas no las aprueba para un motor de otros archivos.
